In [1]:
from nero.teleop.interface import NeroDualArmServer

/home/geist/anaconda3/envs/agilex_teleop/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import logging
log = logging.getLogger(__name__)

In [3]:
import numpy as np
import time

In [4]:
server = NeroDualArmServer(gripper_enabled=True)

[GRIPPER-L] move=0.4ms, total=0.4ms, width=0.000
[GRIPPER-L] move=0.4ms, total=0.4ms, width=0.100
[GRIPPER-R] move=0.4ms, total=0.4ms, width=0.000
[GRIPPER-R] move=0.3ms, total=0.3ms, width=0.100


In [ ]:
left_fp = server.left_robot.get_flange_pose()
left_robot_pose = np.array(left_fp.msg)
right_fp = server.right_robot.get_flange_pose()
right_robot_pose = np.array(right_fp.msg)

print("Left robot z:", left_robot_pose[2])
print("Right robot z:", right_robot_pose[2])

robot z: 0.137929


In [ ]:
assert server.left_robot is not None, "Left arm failed"
assert server.right_robot is not None, "Right arm failed"
print("Left robot:", server.left_robot)
print("Right robot:", server.left_robot)

In [ ]:
left_joints = server.left_robot_get_joint_positions()
right_joints = server.right_robot_get_joint_positions()
print("Left joints:", left_joints)
print("Right joints:", right_joints)

In [ ]:
left_pose = server.left_robot_get_ee_pose()
right_pose = server.right_robot_get_ee_pose()
print("Left pose:", left_pose)
print("Right pose:", right_pose)

In [ ]:
from nero.kinematics.nero_kinematics.nero_ik.ik_solver import fk
from pyAgxArm.utiles.tf import rot_to_rpy

In [ ]:
ik_solver = server.left_ik_solver

T_fk = fk(left_joints, ik_solver.nero_params)
fk_xyz = np.asarray(T_fk[:3, 3], dtype=float)
fk_rpy = np.asarray(rot_to_rpy(T_fk[:3, :3].tolist()), dtype=float)
print("FK XYZ:", fk_xyz)
print("FK RPY:", fk_rpy)

In [ ]:
# left_arm_status = server.left_robot_get_arm_status()
# right_arm_status = server.right_robot_get_arm_status()

# print("Left arm status:", left_arm_status)
# print("Right arm status:", right_arm_status)

In [ ]:
server.left_robot_go_home()
server.right_robot_go_home()

In [ ]:
# left_joints = server.left_robot_get_joint_positions()
# print("Left joints:", left_joints)

# left_joints_target = np.array(left_joints).copy()
# left_joints_target[0] -= np.deg2rad(0)
# left_joints_target[1] += np.deg2rad(0)
# left_joints_target[2] += np.deg2rad(0)
# left_joints_target[3] += np.deg2rad(0)
# left_joints_target[4] += np.deg2rad(0)
# left_joints_target[5] += np.deg2rad(0)
# left_joints_target[6] += np.deg2rad(0)

# print("Left joints target:", left_joints_target)

# server.left_robot_move_to_joint_positions(left_joints_target, delta=False)

# left_joints_real = server.left_robot_get_joint_positions()
# print("Left joints:", left_joints_real)

In [ ]:
# left_pose = server.left_robot_get_ee_pose()
# print("Left pose:", left_pose)

# pose_target = np.array(left_pose).copy()
# # end-effector pose [x, y, z, roll, pitch, yaw] (m, rad)
# # pose_target[0] += 0.01   # x方向移动1cm
# # pose_target[1] += 0.01
# # pose_target[2] += 0.01
# pose_target = [-0.4, 0.0, 0.4, 1.5708, 0.0, 0.0]

# server.left_robot_move_to_ee_pose(pose_target)
# time.sleep(3.0)
# left_pose = server.left_robot_get_ee_pose()
# print("Left pose:", left_pose)

In [ ]:
# # 绝对控制
# target = [0, -70, 0, 100, 0, 0, 30]  # 7维绝对关节角度（度）

# ok = server.servo_j("left_robot", target, delta=False)
# print("servo_j ok:", ok)

In [ ]:
# # 增量控制
# target_delta = [10, 10, 10, 10, 10, 10, 10]

# ok = server.servo_j("left_robot", target_delta, delta=True)
# print("servo_j_delta ok:", ok)

In [ ]:
# 单步 servo_p 测试

# 构造一个小的 delta
# cur_pose = [-0.226,0,0.4,-1.57,0,-3.14]
delta_pose = np.array([-0.0, 0.0, -0.1, 0.0, 0.0, -0.0])
# target_pose = np.array([-0.403, 0.03, 0.265, 1.57, -0.35, -0.07])

# 调用 servo_p（delta 模式）
# server.servo_p_OL(
#     robot_arm="left_robot",
#     pose=delta_pose.tolist(),
#     delta=True
#     # pose=target_pose.tolist(),
#     # delta=False
# )
server.servo_p_OL(
    robot_arm="right_robot",
    pose=delta_pose.tolist(),
    delta=True
    # pose=target_pose.tolist(),
    # delta=False
)

In [7]:
# 连续 servo_p 测试
steps = 50
dt = 0.02  # 20Hz
# cur_pose = [-0.226,0,0.4,-1.57,0,-3.14]
for i in range(steps):
    # delta_pose = np.array([0.01, -0.00125, 0.0125, 0.0, 0.0, 0.0])
    if(i < 25):
        delta_pose = np.array([-0.005, -0.005, 0.0, 0.0, 0.0, 0.0])
    else:
        delta_pose = np.array([0.005, 0.005, 0.0, 0.0, 0.0, 0.0])
    time1 = time.time()
    server.servo_p_OL("left_robot", delta_pose.tolist(), delta=True)
    time2 = time.time()
    print(f"Step {i+1}/{steps}, servo_p_OL time: {(time2 - time1) * 1000:.2f} ms")
    time.sleep(dt)

Step 1/50, servo_p_OL time: 2.75 ms
Step 2/50, servo_p_OL time: 1.37 ms
Step 3/50, servo_p_OL time: 1.63 ms
Step 4/50, servo_p_OL time: 1.18 ms
Step 5/50, servo_p_OL time: 1.28 ms
Step 6/50, servo_p_OL time: 1.56 ms
Step 7/50, servo_p_OL time: 3.14 ms
Step 8/50, servo_p_OL time: 2.65 ms
Step 9/50, servo_p_OL time: 3.12 ms
Step 10/50, servo_p_OL time: 2.59 ms
Step 11/50, servo_p_OL time: 2.41 ms
Step 12/50, servo_p_OL time: 1.46 ms
Step 13/50, servo_p_OL time: 2.43 ms
Step 14/50, servo_p_OL time: 2.44 ms
Step 15/50, servo_p_OL time: 3.15 ms
Step 16/50, servo_p_OL time: 2.54 ms
Step 17/50, servo_p_OL time: 5.78 ms
Step 18/50, servo_p_OL time: 0.80 ms
Step 19/50, servo_p_OL time: 2.41 ms
Step 20/50, servo_p_OL time: 3.37 ms
Step 21/50, servo_p_OL time: 3.75 ms
Step 22/50, servo_p_OL time: 3.36 ms
Step 23/50, servo_p_OL time: 3.10 ms
Step 24/50, servo_p_OL time: 2.75 ms
Step 25/50, servo_p_OL time: 1.85 ms
Step 26/50, servo_p_OL time: 4.01 ms
Step 27/50, servo_p_OL time: 3.70 ms
Step 28/50

In [ ]:
# YZ 平面正方形轨迹测试 - 重复10次
# 正方形边长 0.15m，分为 10 段，每段 0.015m
steps_per_side = 10  # 每边分10步
side_length = 0.15   # 边长 0.15m
step_size = side_length / steps_per_side  # 每步移动距离
dt = 0.02  # 20Hz 控制周期
num_repeats = 10  # 重复次数

print(f"正方形边长: {side_length}m, 每边步数: {steps_per_side}, 每步距离: {step_size}m")
print(f"重复次数: {num_repeats}")

for repeat in range(num_repeats):
    print(f"\n========== 第 {repeat + 1}/{num_repeats} 次正方形 ==========")
    
    # 正方形轨迹: 
    # 边1: y += 0.2 (向前)
    # 边2: z += 0.2 (向上)  
    # 边3: y -= 0.2 (向后)
    # 边4: z -= 0.2 (向下)

    # 边1: y 方向 +0.2m
    print(f"边1: y 方向 +0.2m")
    for i in range(steps_per_side):
        delta_pose = np.array([0.0, step_size, 0.0, 0.0, 0.0, 0.0])
        server.servo_p_OL("left_robot", delta_pose.tolist(), delta=True)
        # server.left_gripper_goto(0.0)
        # time.sleep(dt)

    # 边2: z 方向 +0.2m
    print(f"边2: z 方向 +0.2m")
    for i in range(steps_per_side):
        delta_pose = np.array([0.0, 0.0, step_size, 0.0, 0.0, 0.0])
        server.servo_p_OL("left_robot", delta_pose.tolist(), delta=True)
        # server.left_gripper_goto(0.05)
        # time.sleep(dt)

    # 边3: y 方向 -0.2m
    print(f"边3: y 方向 -0.2m")
    for i in range(steps_per_side):
        delta_pose = np.array([0.0, -step_size, 0.0, 0.0, 0.0, 0.0])
        server.servo_p_OL("left_robot", delta_pose.tolist(), delta=True)
        # server.left_gripper_goto(0.0)
        # time.sleep(dt)

    # 边4: z 方向 -0.2m
    print(f"边4: z 方向 -0.2m")
    for i in range(steps_per_side):
        delta_pose = np.array([0.0, 0.0, -step_size, 0.0, 0.0, 0.0])
        server.servo_p_OL("left_robot", delta_pose.tolist(), delta=True)
        # server.left_gripper_goto(0.1)
        # time.sleep(dt)

print("\n所有正方形轨迹完成!")

In [ ]:
# server.left_gripper_goto(0.05)
# # server.left_gripper_grasp()
# print(server.left_gripper_get_state())

In [ ]:
server.stop("left_robot")